## Setup

Imports and the list of bases coprime with 15.

In [2]:
import math
import random

from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import QFTGate
from qiskit_aer import AerSimulator

SUPPORTED_BASES = [2, 4, 7, 8, 11, 13]

#### Classical helpers

`compute_period` gives us the true period T (we use it to check the quantum result). `continued_fraction_denominators` recovers candidate periods from a measured fraction k/Q.

In [3]:
def compute_period(a, N):
    T = 1
    while pow(a, T, N) != 1:
        T += 1
    return T


def continued_fraction_denominators(num, den, max_value):
    denominators = []
    q_prev_prev, q_prev = 1, 0

    while den != 0:
        a_i = num // den
        q_curr = a_i * q_prev + q_prev_prev
        if q_curr >= max_value:
            break
        if q_curr > 0:
            denominators.append(q_curr)
        q_prev_prev, q_prev = q_prev, q_curr
        num, den = den, num % den

    return denominators

#### Modular multiplication oracle

Controlled-U for `a^power mod 15`. Every base has a set permutation for the 4 qubits.

In [4]:
def controlled_amod15(a, power):
    if a not in SUPPORTED_BASES:
        raise ValueError(f"a must be one of {SUPPORTED_BASES}")

    U = QuantumCircuit(4)
    for _ in range(power):
        if a in (2, 13):
            U.swap(0, 1); U.swap(1, 2); U.swap(2, 3)
        if a in (7, 8):
            U.swap(2, 3); U.swap(1, 2); U.swap(0, 1)
        if a in (11, 4):
            U.swap(1, 3); U.swap(0, 2)
        if a in (7, 11, 13):
            for q in range(4):
                U.x(q)

    return U.to_gate(label=f"{a}^{power} mod 15").control()

#### Shor circuit

Hadamards on the counting register, controlled powers of U, and inverse QFT.

In [5]:
def build_shor_circuit(n_count, n_aux, a):
    qc = QuantumCircuit(n_count + n_aux, n_count)
    for q in range(n_count):
        qc.h(q)
    qc.x(n_count)

    aux_qubits = [i + n_count for i in range(n_aux)]
    for q in range(n_count):
        qc.append(controlled_amod15(a, 2 ** q), [q] + aux_qubits)

    qc.append(QFTGate(n_count).inverse(), range(n_count))
    qc.measure(range(n_count), range(n_count))
    return qc

#### Post-processing

Check if a measurement k leads to the correct period via continued fractions.

In [6]:
def is_successful_measurement(k, Q, T_expected, a, N):
    if k == 0:
        return False
    for T in continued_fraction_denominators(k, Q, N):
        if T == T_expected and T % 2 == 0 and pow(a, T // 2, N) != N - 1:
            return True
    return False

def evaluate_success(counts, n_count, T_expected, a, N, shots):
    Q = 2 ** n_count
    successes = 0
    for bitstring, count in counts.items():
        k = int(bitstring, 2)
        ok = is_successful_measurement(k, Q, T_expected, a, N)
        print(f"k={k:2}  count={count:3}  check={ok}")
        if ok:
            successes += count
    return successes / shots

#### Utilities

Pick a random valid base and compute the noise overhead factor j.

In [7]:
def calculate_overhead_j(p_ideal, p_noisy):
    if p_noisy <= 0 or p_noisy >= 1 or p_ideal >= 1:
        return float('inf')

    numerator = math.log(1 - p_ideal)
    denominator = math.log(1 - p_noisy)

    return numerator / denominator

def pick_valid_base(N, supported):
    while True:
        a = random.randint(2, N - 1)
        if math.gcd(a, N) > 1 or a not in supported:
            continue
        return a

#### Run the circuit

Pick a base, build the circuit, simulate with 1000 shots.

In [8]:
N_COUNT, N_AUX, N_TARGET, SHOTS = 4, 4, 15, 1000

a = pick_valid_base(N_TARGET, SUPPORTED_BASES)
T = compute_period(a, N_TARGET)
print(f"N = {N_TARGET}, a = {a}, period T = {T}")

qc = build_shor_circuit(N_COUNT, N_AUX, a)
simulator = AerSimulator()
counts = simulator.run(transpile(qc, simulator), shots=SHOTS).result().get_counts()

N = 15, a = 4, period T = 2


#### Success rate

Fraction of shots that recover the correct period.

In [9]:
p_success = evaluate_success(counts, N_COUNT, T, a, N_TARGET, SHOTS)
print(f"Success probability: {p_success * 100:.2f}%")

k= 8  count=515  check=True
k= 0  count=485  check=False
Success probability: 51.50%
